<a href="https://colab.research.google.com/github/prabhnoor14/ds2002-fa26/blob/main/notebooks/01-foundations/2026_09_11_%E2%80%94_SQL_Challenge_Set_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [1]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [2]:
q('''
SELECT tracks.title, artists.name, artists.country
FROM tracks
JOIN artists
ON tracks.artist_id = artists.artist_id
''')

,title,name,country
0,Skyline,Nova Waves,US
1,Undertow,Nova Waves,US
2,Foothills,The Blue Ridge,US
3,Aurora,Kestrel,UK
4,Nightfall,Kestrel,UK
5,Sol,Marisol,ES
6,Coastline,The Blue Ridge,US
7,Ridgeline,The Blue Ridge,US
8,Untitled Demo,Kestrel,UK


Explaination: I select from tracks because that’s where the track information is, and it has the artist_id that connects to the artists table. I use title for the track, then get the artist’s name and country from artists. I use a regular JOIN because every track has an artist in this data, so there’s no need to keep tracks without a matching artist. I expect 9 rows because there are 9 tracks, and I got 9 rows. The untagged track, "Untitled Demo," and the unplayed tracks are still included because this query doesn’t filter by genre or require anything from the plays table.

### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [3]:
q('''
SELECT genre, AVG(seconds)
FROM tracks
GROUP BY genre
ORDER BY AVG(seconds) DESC
LIMIT 1
''')

,genre,AVG(seconds)
0,Electronic,287.5


Explaination: I group the tracks by genre and use AVG(seconds) to find the average length for each genre. I order the averages from highest to lowest and use LIMIT 1 to get the genre with the longest average. I expect 1 row because the question asks for only the genre with the longest average, and I got 1 row. The untagged track has a NULL genre, so it forms its own group but doesn’t affect the answer because AVG ignores the NULL value. The unplayed tracks are still included because this query only uses tracks and doesn’t filter based on whether a track appears in plays.

### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [4]:
q('''
SELECT user, COUNT(play_id) AS plays, COUNT(DISTINCT track_id) AS tracks
FROM plays
GROUP BY user
''')

,user,plays,tracks
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


Explaination: I select from plays because that table has the user, play, and track information. I use COUNT(play_id) to count how many times each user played something, and COUNT(DISTINCT track_id) to see how many different tracks they played. I group by user so each user gets their own row. I expect 4 rows because there are 4 users, and I got 4. The untagged and unplayed tracks don’t affect this because I’m only looking at tracks that actually have a play record, so "Untitled Demo" and the other unplayed tracks are not included.

### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [5]:
q('''
SELECT tracks.title
FROM tracks
LEFT JOIN plays
ON tracks.track_id = plays.track_id
WHERE plays.track_id IS NULL
''')

,title
0,Ridgeline
1,Untitled Demo


Explaination: I start with tracks because I want to find tracks that don't have any plays. I use a LEFT JOIN so that every track stays in the result, even if there is no matching row in plays. Then I use WHERE plays.track_id IS NULL to keep only the tracks that had no match. I expect 2 rows, and I got 2: Ridgeline and Untitled Demo. This specifically handles the unplayed tracks, including the untagged Untitled Demo, instead of accidentally leaving them out.

### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [6]:
q('''
SELECT artists.name, SUM(tracks.seconds) AS total_seconds, ROUND(SUM(tracks.seconds) / 60.0, 1) AS total_minutes
FROM artists
JOIN tracks
ON artists.artist_id = tracks.artist_id
JOIN plays
ON tracks.track_id = plays.track_id
GROUP BY artists.name
ORDER BY total_seconds DESC
 ''')

,name,total_seconds,total_minutes
0,Kestrel,1175,19.6
1,Nova Waves,843,14.1
2,The Blue Ridge,384,6.4
3,Marisol,210,3.5


Explaination: I start with artists and join it to tracks using artist_id, then join plays so I only count tracks that were actually played. I use SUM(tracks.seconds) to get the total listening time for each artist, then divide by 60 and use ROUND(..., 1) for minutes. I group by artist and order by total seconds from highest to lowest. I expect 4 rows because all 4 artists have at least one played track, and I got 4. The untagged and unplayed tracks are important here because the unplayed tracks don't contribute any listening time, while the untagged track is also unplayed, so neither affects the totals.



```
# This is formatted as code
```

### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [7]:
q('''
SELECT track_id, title
FROM tracks
WHERE genre IS NULL
''')

,track_id,title
0,18,Untitled Demo


Explaination: I use WHERE genre IS NULL because I’m looking specifically for tracks where the genre is missing. I expect 1 row, and I got 1: track 18, Untitled Demo. The unplayed status doesn’t matter here because I’m checking the tracks table directly. WHERE genre != 'Pop' would not include Untitled Demo because NULL != 'Pop' is not true in SQL, so the row would be filtered out. This is why IS NULL is needed when checking for missing values.

### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [8]:
q('''
SELECT played_on, count(play_id) AS plays, count(DISTINCT user) AS users
FROM plays
GROUP BY played_on
ORDER BY played_on
''')

,played_on,plays,users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


Explaination: I select from plays because that table has the date, user, and play information. I use COUNT(play_id) for the total plays and COUNT(DISTINCT user) for the number of different users active that day. I group by played_on so I get one row for each date, then order by the date to put them earliest first. I expect 6 rows because there are 6 different dates in the data, and I got 6. The untagged and unplayed tracks don't show up because this is based only on actual rows in plays, so only tracks that were played are counted.

### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [9]:
q1 = q('''
SELECT tracks.title, artists.name, artists.country
FROM tracks
JOIN artists
ON tracks.artist_id = artists.artist_id
''')
q2 = q('''
SELECT genre, AVG(seconds)
FROM tracks
GROUP BY genre
ORDER BY AVG(seconds) DESC
LIMIT 1
''')
q3 = q('''
SELECT user, COUNT(play_id) AS plays, COUNT(DISTINCT track_id) AS tracks
FROM plays
GROUP BY user
''')
q4 = q('''
SELECT tracks.title
FROM tracks
LEFT JOIN plays
ON tracks.track_id = plays.track_id
WHERE plays.track_id IS NULL
''')
assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

Q5 gave me the most trouble because I had two joins and had to figure out how the tables connected. My mistake was initially trying to sum the listening time from the `plays` table, but the track length is actually stored in `tracks.seconds`. I had to join `artists` to `tracks` and then `tracks` to `plays` so I could use the artist information, track length, and actual play records together. Once I realized that, I used `SUM(tracks.seconds)` and grouped by artist to get the total listening time.
